In [ ]:
import pandas as pd

eval_df = pd.read_json("../data/processed/eval_v10.jsonl", lines=True)
eval_df.head()

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../models/Ministral-3-8B-Instruct-2512-GEC-v9", # Point directly to the ADAPTER folder
    max_seq_length = 512,          # Use the same as your training
    load_in_4bit = True,            # Set to True if you trained with 4-bit
    dtype = None,                   # Auto-detect (Float16/Bfloat16)
    device_map="cpu"
)

# Switch model to inference mode (merges LoRA weights for generation)
FastLanguageModel.for_inference(model)

In [ ]:
from src.prompts import get_inference_prompt_v5

eval_df["inference_prompt"] = eval_df["corrupted"].apply(get_inference_prompt_v5)
eval_df.iloc[0]["inference_prompt"]

In [ ]:
# tokenize the list of prompts
tensors = tokenizer(None, eval_df["inference_prompt"].tolist(), return_tensors="pt", add_special_tokens=False, padding=True).to(model.device)
print(tensors["input_ids"].shape)

In [ ]:
from tqdm import tqdm
corrections = []

# Generate corrections for each prompt in batch of 8
for i in tqdm(range(0, len(tensors["input_ids"]), 8), desc="Generating corrections"):
    batch = {key: val[i:i+8] for key, val in tensors.items()}
    output = model.generate(**batch, max_new_tokens=128)

    for j in range(output.shape[0]):
        # Decode only the NEW tokens (skip the input prompt tokens)
        input_length = batch["input_ids"].shape[1]
        generated_tokens = output[j][input_length:]
        output_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        corrections.append(output_text)

In [ ]:
corrections

In [ ]:
pd.DataFrame({
    "corrupted": eval_df["corrupted"],
    "model_corrected": corrections,
    "original": eval_df["original"],
}).to_json("../outputs/Text-Tune-Base-v9-on-eval-v10.jsonl", orient="records", lines=True, force_ascii=False)